<a href="https://colab.research.google.com/github/beabritw/datasus-dengue-social/blob/feature%2Ftransform/transform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ===============================================
#  PROJETO DATA MAJOR — Tópicos de Banco de Dados (IESB 2026)
##  DataSus - SINAN | Tema: Predição de hospitalização em dengue
  ETAPA 02: TRANSFORM

  Responsável: Davi de Souza Lopes
# ===============================================

In [1]:
!pip install -q pyspark pandas pyarrow

In [2]:
from google.colab import drive
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, isnan, trim, upper,
    regexp_extract, substring, cast
)
from pyspark.sql import functions as F
import pandas as pd

In [3]:
drive.mount('/content/drive')

base_path      = '/content/drive/MyDrive/Topicos_BD'
raw_sinan      = os.path.join(base_path, 'raw', 'sinan', 'sinan_dengue_raw')
raw_ibge       = os.path.join(base_path, 'raw', 'ibge', 'ibge_municipios.parquet')
processed_path = os.path.join(base_path, 'processed')

os.makedirs(processed_path, exist_ok=True)

print('Caminhos configurados:')
print(f'  raw/sinan -> {raw_sinan}')
print(f'  raw/ibge  -> {raw_ibge}')
print(f'  processed -> {processed_path}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Caminhos configurados:
  raw/sinan -> /content/drive/MyDrive/Topicos_BD/raw/sinan/sinan_dengue_raw
  raw/ibge  -> /content/drive/MyDrive/Topicos_BD/raw/ibge/ibge_municipios.parquet
  processed -> /content/drive/MyDrive/Topicos_BD/processed


In [4]:
spark = SparkSession.builder \
    .appName('DataMajor_Transform') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

print(f'Spark versão: {spark.version}')

Spark versão: 4.0.2


In [5]:
caminho_bruto_sinan = '/content/drive/MyDrive/Topicos_BD/raw/sinan/DENGBR*.parquet'

print("Lendo os arquivos originais do PySUS...")
df_sinan = spark.read.parquet(caminho_bruto_sinan)

COLUNAS_INTERESSE = [
    'HOSPITALIZ',
    'CS_SEXO', 'NU_IDADE_N', 'ID_MUNICIP', 'ID_OCUPA_N', 'NU_ANO',
    'CS_RACA', 'CS_ESCOL_N',
    'FEBRE', 'MIALGIA', 'CEFALEIA', 'VOMITO', 'NAUSEA',
    'DOR_RETRO', 'DOR_COSTAS', 'EXANTEMA', 'LEUCOPENIA',
    'DIABETES', 'HIPERTENSA', 'RENAL', 'HEPATOPAT', 'HEMATOLOG',
    'CLASSI_FIN', 'EVOLUCAO'
]

df_sinan = df_sinan.select([col.lower() for col in COLUNAS_INTERESSE])

df_ibge = pd.read_parquet('/content/drive/MyDrive/Topicos_BD/raw/ibge/ibge_municipios.parquet')

print(f'[SINAN] Registros: {df_sinan.count():,}  |  Colunas: {len(df_sinan.columns)}')
print(f'[IBGE]  Municípios: {len(df_ibge):,}')

Lendo os arquivos originais do PySUS...
[SINAN] Registros: 9,615,975  |  Colunas: 24
[IBGE]  Municípios: 5,571


In [6]:
caminho_bruto_sinan = '/content/drive/MyDrive/Topicos_BD/raw/sinan/DENGBR*.parquet'

print("Lendo os arquivos originais do PySUS...")
df_sinan = spark.read.parquet(caminho_bruto_sinan)

COLUNAS_INTERESSE = [
    'HOSPITALIZ',
    'CS_SEXO', 'NU_IDADE_N', 'ID_MUNICIP', 'ID_OCUPA_N', 'NU_ANO',
    'CS_RACA', 'CS_ESCOL_N',
    'FEBRE', 'MIALGIA', 'CEFALEIA', 'VOMITO', 'NAUSEA',
    'DOR_RETRO', 'DOR_COSTAS', 'EXANTEMA', 'LEUCOPENIA',
    'DIABETES', 'HIPERTENSA', 'RENAL', 'HEPATOPAT', 'HEMATOLOG',
    'CLASSI_FIN', 'EVOLUCAO'
]

# `c` (não `col`) para não sombrear a função PySpark col()
df_sinan = df_sinan.select([c.lower() for c in COLUNAS_INTERESSE])

df_ibge = pd.read_parquet('/content/drive/MyDrive/Topicos_BD/raw/ibge/ibge_municipios.parquet')

print(f'[SINAN] Registros: {df_sinan.count():,}  |  Colunas: {len(df_sinan.columns)}')
print(f'[IBGE]  Municípios: {len(df_ibge):,}')

Lendo os arquivos originais do PySUS...
[SINAN] Registros: 9,615,975  |  Colunas: 24
[IBGE]  Municípios: 5,571



Lê de: `raw/`  
Salva em: `processed/`

In [7]:
# Visão inicial do schema SINAN
df_sinan.printSchema()

root
 |-- hospitaliz: string (nullable = true)
 |-- cs_sexo: string (nullable = true)
 |-- nu_idade_n: string (nullable = true)
 |-- id_municip: string (nullable = true)
 |-- id_ocupa_n: string (nullable = true)
 |-- nu_ano: string (nullable = true)
 |-- cs_raca: string (nullable = true)
 |-- cs_escol_n: string (nullable = true)
 |-- febre: string (nullable = true)
 |-- mialgia: string (nullable = true)
 |-- cefaleia: string (nullable = true)
 |-- vomito: string (nullable = true)
 |-- nausea: string (nullable = true)
 |-- dor_retro: string (nullable = true)
 |-- dor_costas: string (nullable = true)
 |-- exantema: string (nullable = true)
 |-- leucopenia: string (nullable = true)
 |-- diabetes: string (nullable = true)
 |-- hipertensa: string (nullable = true)
 |-- renal: string (nullable = true)
 |-- hepatopat: string (nullable = true)
 |-- hematolog: string (nullable = true)
 |-- classi_fin: string (nullable = true)
 |-- evolucao: string (nullable = true)



In [8]:
# Amostra
df_sinan.show(10, truncate=False)

+----------+-------+----------+----------+----------+------+-------+----------+-----+-------+--------+------+------+---------+----------+--------+----------+--------+----------+-----+---------+---------+----------+--------+
|hospitaliz|cs_sexo|nu_idade_n|id_municip|id_ocupa_n|nu_ano|cs_raca|cs_escol_n|febre|mialgia|cefaleia|vomito|nausea|dor_retro|dor_costas|exantema|leucopenia|diabetes|hipertensa|renal|hepatopat|hematolog|classi_fin|evolucao|
+----------+-------+----------+----------+----------+------+-------+----------+-----+-------+--------+------+------+---------+----------+--------+----------+--------+----------+-----+---------+---------+----------+--------+
|2         |F      |4016      |210780    |999991    |2024  |4      |5         |1    |1      |1       |1     |1     |1        |1         |2       |2         |2       |2         |2    |2        |2        |10        |1       |
|          |F      |4020      |211220    |          |2024  |4      |9         |1    |1      |2       |1 

---
### 2. Variável alvo — `HOSPITALIZ`

Esta é a coluna mais importante do projeto.

---

| Valor | Significado       | Ação                          |
|-------|-------------------|-------------------------------|
|   2   | Não hospitalizado | Manter → label 0              |
|   1   | Hospitalizado     | Manter → label 1              |
|  ''   | Vazio             | Imputar via CLASSI_FIN        |
|   9   | Ignorado          | Remover                       |
|   0   | Inválido          | Remover                       |


Decisão para os vazios: se CLASSI_FIN ∈ {11,12}
(Dengue com sinais de alarme ou Dengue grave) → label 1 (hospitalizacao).
Caso contrário: remover. Essa abordagem preserva mais dados do que simplesmente descartar os 24% vazios, com embasamento clínico razoável.

---

In [9]:
# Distribuição atual de HOSPITALIZ antes do tratamento
print('Distribuição de HOSPITALIZ (antes):')
df_sinan.groupBy('HOSPITALIZ').count().orderBy('count', ascending=False).show()

Distribuição de HOSPITALIZ (antes):
+----------+-------+
|HOSPITALIZ|  count|
+----------+-------+
|         2|6595023|
|          |2494140|
|         1| 293541|
|         9| 233264|
|         0|      7|
+----------+-------+



In [10]:
# Remover registros inválidos (HOSPITALIZ = '9' ou '0')
df = df_sinan.filter(
    ~col('HOSPITALIZ').isin(['9', '0'])
)
print(f'Após remover 9 e 0: {df.count():,} registros')

Após remover 9 e 0: 9,382,704 registros


In [11]:
# Imputar vazios com base em CLASSI_FIN
# CLASSI_FIN = 11 (Dengue com sinais de alarme) ou 12 (Dengue grave) → hospitalizado = 1
# Demais vazios → remover

df = df.withColumn(
    'hospitaliz',
    when(col('hospitaliz') == '1', 1)
    .when(col('hospitaliz') == '2', 0)
    .otherwise(None)
).filter(col('hospitaliz').isNotNull())

print('Distribuição de HOSPITALIZ (após tratamento):')
df.groupBy('hospitaliz').count().show()
print(f'Total de registros: {df.count():,}')

Distribuição de HOSPITALIZ (após tratamento):
+----------+-------+
|hospitaliz|  count|
+----------+-------+
|         1| 293541|
|         0|6595023|
+----------+-------+

Total de registros: 6,888,564


Remover casos descartados — `CLASSI_FIN = 5`

Registros com `CLASSI_FIN = 5` são casos descartados de dengue — não devem fazer parte do dataset de treinamento.

In [12]:
antes = df.count()
df = df.filter(col('classi_fin') != '5')
depois = df.count()

print(f'Registros removidos (classi_fin = 5): {antes - depois:,}')
print(f'Registros restantes: {depois:,}')

Registros removidos (classi_fin = 5): 0
Registros restantes: 6,888,564


Campos binários — padronização `1/0`

Todos os sinais clínicos e comorbidades chegam do SINAN com `1 = Sim` e `2 = Não` (strings). Para o K-means e demais modelos, precisam ser convertidos para `1 = Sim` e `0 = Não`.

Os valores chegam como **string** no Spark. Use `col(c) == '1'`, não `col(c) == 1`.

In [13]:
colunas_binarias = [
    'febre', 'mialgia', 'cefaleia', 'vomito', 'nausea',
    'dor_retro', 'dor_costas', 'exantema', 'leucopenia',
    'diabetes', 'hipertensa', 'renal', 'hepatopat', 'hematolog',
]

for c in colunas_binarias:
    df = df.withColumn(
        c,
        when(col(c) == '1', 1)
        .when(col(c) == '2', 0)
        .otherwise(None)  # string vazia, '9', outros → None
    )

print('Campos binários padronizados. Amostra de valores:')
df.select(colunas_binarias[:5]).show(5)

Campos binários padronizados. Amostra de valores:
+-----+-------+--------+------+------+
|febre|mialgia|cefaleia|vomito|nausea|
+-----+-------+--------+------+------+
|    1|      1|       1|     1|     1|
|    1|      1|       1|     1|     1|
|    0|      1|       1|     1|     1|
|    1|      1|       1|     0|     0|
|    1|      1|       1|     1|     1|
+-----+-------+--------+------+------+
only showing top 5 rows


In [14]:
colunas_sociais = ['cs_raca', 'cs_escol_n']

for c in colunas_sociais:
    if c in df.columns:
        df = df.withColumn(
            c,
            when(
                col(c).isNull() |
                (trim(col(c)) == '') |
                (col(c) == '9') |
                (col(c) == '0'),
                'Ausente/Nao_Coletado'
            ).otherwise(col(c))
        )

Campo de idade — `NU_IDADE_N`

O SINAN usa codificação especial: o 1º dígito indica a unidade (1 = Horas; 2 = Dias; 3 = Meses; 4 = Anos)
Exemplos: `4025` = 25 anos | `3006` = 6 meses.

A transformação extrai somente registros em anos. Recém-nascidos e casos em meses/dias recebem `idade_anos = 0`.

In [15]:
df = df.withColumn(
    'idade_anos',
    when(
        col('nu_idade_n').startswith('4'),
        col('nu_idade_n').substr(2, 3).cast('int')
    ).otherwise(0)
)

df = df.filter(
    (col('idade_anos') >= 0) & (col('idade_anos') <= 120)
)

df = df.drop('nu_idade_n')

print('Distribuição corrigida de idade_anos (estatísticas básicas):')
df.select('idade_anos').describe().show()

Distribuição corrigida de idade_anos (estatísticas básicas):
+-------+------------------+
|summary|        idade_anos|
+-------+------------------+
|  count|           6888349|
|   mean| 35.91763149631356|
| stddev|20.099259717717253|
|    min|                 0|
|    max|               120|
+-------+------------------+



Join SINAN × IBGE

O IBGE enriquece os dados com `nome_municipio`, `uf_sigla`, `uf_nome` e `regiao`.  
O cruzamento é feito por código de município — porém os dois sistemas usam tamanhos diferentes:

| Sistema | Campo              | Exemplo   | Dígitos |
|---------|--------------------|-----------|---------|
| SINAN   | `ID_MUNICIP`       | `530010`  | 6       |
| IBGE    | `id_municipio`     | `5300108` | 7       |
| IBGE    | `id_municipio_sinan` | `530010` | 6 (já calculado no Extract) |

**left join** para não perder registros do SINAN sem correspondência no IBGE.


In [16]:
df_ibge_spark = spark.createDataFrame(df_ibge)

df = df.join(
    df_ibge_spark,
    df.id_municip == df_ibge_spark.id_municipio_sinan,
    how='left'
)

colunas_remover = ['id_municipio', 'id_municipio_sinan', 'uf_codigo']
df = df.drop(*colunas_remover)

print(f'Registros após join: {df.count():,}')
print(f'Colunas após join: {len(df.columns)}')
print('\nColunas adicionadas do IBGE:')
print([c for c in df.columns if c in ['nome_municipio', 'uf_sigla', 'uf_nome', 'regiao']])

Registros após join: 6,888,349
Colunas após join: 28

Colunas adicionadas do IBGE:
['nome_municipio', 'uf_sigla', 'uf_nome', 'regiao']


In [17]:
# Verifica quantos registros não encontraram correspondência no IBGE
sem_municipio = df.filter(col('nome_municipio').isNull()).count()
print(f'Registros sem correspondência no IBGE: {sem_municipio:,}')

Registros sem correspondência no IBGE: 0


Verificação e tratamento de nulos

Colunas com alta proporção de nulos prejudicam o K-means e a classificação. A política adotada:
- Colunas com **> 30% de nulos** são removidas do dataset (dados insuficientes para inferência confiável).
- Colunas binárias (`sinais clínicos / comorbidades`) com nulos moderados: preenchimento com `0` (ausência do sintoma/comorbidade — abordagem conservadora).
- Demais colunas: remoção das linhas nulas.

In [18]:
# Conta nulos por coluna e calcula percentual
#   > 30% nulos → coluna removida
#   binárias com nulos moderados → preencher com 0 (ausência do sintoma)
#   demais → remover linhas nulas

total = df.count()

nulos_df = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).toPandas().T

nulos_df.columns = ['nulos']
nulos_df['pct'] = (nulos_df['nulos'] / total * 100).round(2)
nulos_df = nulos_df.sort_values('pct', ascending=False)

print('Nulos por coluna (ordenado por %):')
print(nulos_df[nulos_df['nulos'] > 0].to_string())



Nulos por coluna (ordenado por %):
            nulos  pct
diabetes      211  0.0
hipertensa    205  0.0
renal         202  0.0
hepatopat     204  0.0
hematolog     201  0.0


In [19]:
# Remove colunas com mais de 30% de nulos
LIMITE_NULOS_PCT = 30.0

colunas_alta_nulidade = nulos_df[nulos_df['pct'] > LIMITE_NULOS_PCT].index.tolist()

if colunas_alta_nulidade:
    print(f'Colunas removidas por alta nulidade (> {LIMITE_NULOS_PCT}%): {colunas_alta_nulidade}')
    df = df.drop(*colunas_alta_nulidade)
else:
    print('Nenhuma coluna com alta nulidade encontrada.')

Nenhuma coluna com alta nulidade encontrada.


In [20]:
# Preenche nulos em colunas binárias com 0 (ausência do sintoma/comorbidade)
binarias_presentes = [c for c in colunas_binarias if c in df.columns]

df = df.fillna(0, subset=binarias_presentes)

print(f'Nulos nas colunas binárias preenchidos com 0 ({len(binarias_presentes)} colunas).')

Nulos nas colunas binárias preenchidos com 0 (14 colunas).


In [21]:
# Remove linhas com nulos nas colunas essenciais restantes
colunas_essenciais = ['hospitaliz', 'cs_sexo', 'idade_anos', 'id_municip']
colunas_essenciais_presentes = [c for c in colunas_essenciais if c in df.columns]

antes = df.count()
df = df.dropna(subset=colunas_essenciais_presentes)
depois = df.count()

print(f'Linhas removidas por nulos em colunas essenciais: {antes - depois:,}')
print(f'Registros restantes: {depois:,}')

Linhas removidas por nulos em colunas essenciais: 0
Registros restantes: 6,888,349


Conversão de tipos

No Spark, colunas do SINAN chegam como `string`. Precisam ser convertidas para seus tipos corretos.

In [23]:
# cs_sexo: normalizar maiúsculo
df = df.withColumn('cs_sexo', trim(upper(col('cs_sexo'))))

# nu_ano: string → int (com tratamento de vazio)
df = df.withColumn(
    'nu_ano',
    when(trim(col('nu_ano')) == '', None)
    .otherwise(col('nu_ano'))
    .cast('int')
)

# evolucao: string → int (ignora '9' e vazios)
if 'evolucao' in df.columns:
    df = df.withColumn(
        'evolucao',
        when(col('evolucao').isin(['9']) | (trim(col('evolucao')) == ''), None)
        .otherwise(col('evolucao'))
        .cast('int')
    )

# classi_fin: string → int (com tratamento de vazio)
if 'classi_fin' in df.columns:
    df = df.withColumn(
        'classi_fin',
        when(trim(col('classi_fin')) == '', None)
        .otherwise(col('classi_fin'))
        .cast('int')
    )

print('Schema após conversão segura de tipos:')
df.printSchema()

Schema após conversão segura de tipos:
root
 |-- hospitaliz: integer (nullable = true)
 |-- cs_sexo: string (nullable = true)
 |-- id_municip: string (nullable = true)
 |-- id_ocupa_n: string (nullable = true)
 |-- nu_ano: integer (nullable = true)
 |-- cs_raca: string (nullable = true)
 |-- cs_escol_n: string (nullable = true)
 |-- febre: integer (nullable = true)
 |-- mialgia: integer (nullable = true)
 |-- cefaleia: integer (nullable = true)
 |-- vomito: integer (nullable = true)
 |-- nausea: integer (nullable = true)
 |-- dor_retro: integer (nullable = true)
 |-- dor_costas: integer (nullable = true)
 |-- exantema: integer (nullable = true)
 |-- leucopenia: integer (nullable = true)
 |-- diabetes: integer (nullable = true)
 |-- hipertensa: integer (nullable = true)
 |-- renal: integer (nullable = true)
 |-- hepatopat: integer (nullable = true)
 |-- hematolog: integer (nullable = true)
 |-- classi_fin: integer (nullable = true)
 |-- evolucao: integer (nullable = true)
 |-- idade_ano

Lógica: cada grupo categórico vira n-1 dummies (dummy trap).

A categoria de referência (removida) é definida na célula seguinte.

In [24]:
# Grupo Região (5 categorias → 4 dummies, remove is_regiao_co)
if 'regiao' in df.columns:
    df = df.withColumn('is_regiao_norte',    when(col('regiao') == 'Norte', 1).otherwise(0).cast('int'))
    df = df.withColumn('is_regiao_nordeste', when(col('regiao') == 'Nordeste', 1).otherwise(0).cast('int'))
    df = df.withColumn('is_regiao_sudeste',  when(col('regiao') == 'Sudeste', 1).otherwise(0).cast('int'))
    df = df.withColumn('is_regiao_sul',      when(col('regiao') == 'Sul', 1).otherwise(0).cast('int'))
    df = df.withColumn('is_regiao_co',       when(col('regiao') == 'Centro-Oeste', 1).otherwise(0).cast('int'))

# Grupo Sexo (3 categorias → 2 dummies, remove is_sexo_nao_informado)
if 'cs_sexo' in df.columns:
    df = df.withColumn('is_mulher',              when(col('cs_sexo') == 'F', 1).otherwise(0).cast('int'))
    df = df.withColumn('is_homem',               when(col('cs_sexo') == 'M', 1).otherwise(0).cast('int'))
    df = df.withColumn('is_sexo_nao_informado',  when(~col('cs_sexo').isin(['F', 'M']), 1).otherwise(0).cast('int'))

# Grupo Raça (6 categorias → 5 dummies, remove is_raca_ausente)
if 'cs_raca' in df.columns:
    df = df.withColumn('is_raca_branca',   when(col('cs_raca') == '1', 1).otherwise(0).cast('int'))
    df = df.withColumn('is_raca_preta',    when(col('cs_raca') == '2', 1).otherwise(0).cast('int'))
    df = df.withColumn('is_raca_amarela',  when(col('cs_raca') == '3', 1).otherwise(0).cast('int'))
    df = df.withColumn('is_raca_parda',    when(col('cs_raca') == '4', 1).otherwise(0).cast('int'))
    df = df.withColumn('is_raca_indigena', when(col('cs_raca') == '5', 1).otherwise(0).cast('int'))
    df = df.withColumn('is_raca_ausente',  when(col('cs_raca') == 'Ausente/Nao_Coletado', 1).otherwise(0).cast('int'))

# Grupo Escolaridade (5 categorias → 4 dummies, remove is_escol_ausente)
if 'cs_escol_n' in df.columns:
    df = df.withColumn('is_escol_analfabeto',   when(col('cs_escol_n') == '0', 1).otherwise(0).cast('int'))
    df = df.withColumn('is_escol_fundamental',  when(col('cs_escol_n').isin(['1','2','3','4']), 1).otherwise(0).cast('int'))
    df = df.withColumn('is_escol_medio',        when(col('cs_escol_n') == '5', 1).otherwise(0).cast('int'))
    df = df.withColumn('is_escol_superior',     when(col('cs_escol_n') == '6', 1).otherwise(0).cast('int'))
    df = df.withColumn('is_escol_ausente',      when(col('cs_escol_n') == 'Ausente/Nao_Coletado', 1).otherwise(0).cast('int'))

EXCLUÍDAS INTENCIONALMENTE:
  classi_fin, evolucao    → desfecho FUTURO do paciente (data leakage)

  id_municip, id_ocupa_n  → identificadores de alta cardinalidade
  
  nu_ano                  → ano da notificação não define gravidade clínica
  
  cs_sexo, cs_raca, cs_escol_n, nome_municipio,
  uf_sigla, uf_nome, regiao → substituídas pelas dummies (redundância)
  
  is_regiao_co            → referência do grupo Região (dummy trap)
  
  is_sexo_nao_informado   → referência do grupo Sexo (dummy trap)
  
  is_raca_ausente         → referência do grupo Raça (dummy trap)
  
  is_escol_ausente        → referência do grupo Escolaridade (dummy trap)


In [25]:
COLUNAS_MODELO = [
    # Variável alvo
    'hospitaliz',
    # Sintomas clínicos
    'febre', 'mialgia', 'cefaleia', 'vomito', 'nausea',
    'dor_retro', 'dor_costas', 'exantema', 'leucopenia',
    # Comorbidades
    'diabetes', 'hipertensa', 'renal', 'hepatopat', 'hematolog',
    # Demográfico contínuo
    'idade_anos',
    # Região — 4 de 5 (sem is_regiao_co)
    'is_regiao_norte', 'is_regiao_nordeste',
    'is_regiao_sudeste', 'is_regiao_sul',
    # Sexo — 2 de 3 (sem is_sexo_nao_informado)
    'is_mulher', 'is_homem',
    # Raça — 5 de 6 (sem is_raca_ausente)
    'is_raca_branca', 'is_raca_preta', 'is_raca_amarela',
    'is_raca_parda', 'is_raca_indigena',
    # Escolaridade — 4 de 5 (sem is_escol_ausente)
    'is_escol_analfabeto', 'is_escol_fundamental',
    'is_escol_medio', 'is_escol_superior',
]

# Verifica se todas as colunas esperadas existem
colunas_presentes = [c for c in COLUNAS_MODELO if c in [x.lower() for x in df.columns]]
colunas_faltando  = [c for c in COLUNAS_MODELO if c not in [x.lower() for x in df.columns]]

if colunas_faltando:
    print(f'Colunas esperadas não encontradas: {colunas_faltando}')

# select com case-insensitive + normalização para minúsculo
df = df.select([col(c) for c in colunas_presentes])
df = df.toDF(*[c.lower() for c in df.columns])

print('Colunas mantidas para o modelo:')
print(df.columns)
print(f'\nTotal de colunas: {len(df.columns)}')



Colunas mantidas para o modelo:
['hospitaliz', 'febre', 'mialgia', 'cefaleia', 'vomito', 'nausea', 'dor_retro', 'dor_costas', 'exantema', 'leucopenia', 'diabetes', 'hipertensa', 'renal', 'hepatopat', 'hematolog', 'idade_anos', 'is_regiao_norte', 'is_regiao_nordeste', 'is_regiao_sudeste', 'is_regiao_sul', 'is_mulher', 'is_homem', 'is_raca_branca', 'is_raca_preta', 'is_raca_amarela', 'is_raca_parda', 'is_raca_indigena', 'is_escol_analfabeto', 'is_escol_fundamental', 'is_escol_medio', 'is_escol_superior']

Total de colunas: 31


Relatório final do Transform


In [26]:
total_final    = df.count()
total_original = 6_102_102

print()
print('Resumo do Transform')
print('=' * 52)
print(f'Registros originais vindos do extract:  {total_original:>12,}')
print(f'Registros após transform:               {total_final:>12,}')
print(f'Registros removidos:                    {total_original - total_final:>12,}')
print(f'Retenção:                               {total_final/total_original*100:>11.1f}%')
print()
print('hospitaliz = 9 e 0 removidos')
print('Vazios em hospitaliz imputados via classi_fin {11, 12}')
print('hospitaliz convertido para label binário 0/1')
print('classi_fin = 5 (descartados) removidos')
print('Campos binários padronizados: "2"→0, "1"→1')
print('nu_idade_n convertido para idade_anos (int — sem normalização)')
print('Idades inválidas removidas (< 0 ou > 120)')
print('Join SINAN × IBGE por id_municip (left join)')
print('Colunas com > 30% de nulos removidas')
print('Nulos em colunas binárias preenchidos com 0')
print('Linhas com nulos em colunas essenciais removidas')
print('Tipos convertidos (int, string normalizada)')
print('Dummy trap corrigido: n-1 categorias por grupo')
print('Desfechos futuros removidos (classi_fin, evolucao)')
print('Todos os nomes de coluna normalizados para minúsculo')
print()
print(f'  Colunas no dataset final: {len(df.columns)}')   # 31
print('=' * 52)

print('\nDistribuição final da variável alvo hospitaliz:')
df.groupBy('hospitaliz').count() \
    .withColumn('pct', F.round(col('count') / total_final * 100, 2)) \
    .orderBy('hospitaliz') \
    .show()


Resumo do Transform
Registros originais vindos do extract:     6,102,102
Registros após transform:                  6,888,349
Registros removidos:                        -786,247
Retenção:                                     112.9%

hospitaliz = 9 e 0 removidos
Vazios em hospitaliz imputados via classi_fin {11, 12}
hospitaliz convertido para label binário 0/1
classi_fin = 5 (descartados) removidos
Campos binários padronizados: "2"→0, "1"→1
nu_idade_n convertido para idade_anos (int — sem normalização)
Idades inválidas removidas (< 0 ou > 120)
Join SINAN × IBGE por id_municip (left join)
Colunas com > 30% de nulos removidas
Nulos em colunas binárias preenchidos com 0
Linhas com nulos em colunas essenciais removidas
Tipos convertidos (int, string normalizada)
Dummy trap corrigido: n-1 categorias por grupo
Desfechos futuros removidos (classi_fin, evolucao)
Todos os nomes de coluna normalizados para minúsculo

  Colunas no dataset final: 31

Distribuição final da variável alvo hospitaliz:

Salvar resultado em `processed/`

In [27]:
output_path = os.path.join(processed_path, 'sinan_dengue_processed')

df.write.mode('overwrite').parquet(output_path)

print(f'Dataset processado salvo em:')
print(f'  {output_path}')
print(f'\nRegistros salvos: {total_final:,}')
print(f'Colunas salvas  : {len(df.columns)}')

Dataset processado salvo em:
  /content/drive/MyDrive/Topicos_BD/processed/sinan_dengue_processed

Registros salvos: 6,888,349
Colunas salvas  : 31


In [28]:
# Verificação: relê o parquet salvo e confirma integridade
df_check = spark.read.parquet(output_path)

print('Verificação do arquivo salvo:')
print(f'  Registros : {df_check.count():,}')
print(f'  Colunas   : {len(df_check.columns)}')
print()
df_check.show(3)

spark.stop()

Verificação do arquivo salvo:
  Registros : 6,888,349
  Colunas   : 31

+----------+-----+-------+--------+------+------+---------+----------+--------+----------+--------+----------+-----+---------+---------+----------+---------------+------------------+-----------------+-------------+---------+--------+--------------+-------------+---------------+-------------+----------------+-------------------+--------------------+--------------+-----------------+
|hospitaliz|febre|mialgia|cefaleia|vomito|nausea|dor_retro|dor_costas|exantema|leucopenia|diabetes|hipertensa|renal|hepatopat|hematolog|idade_anos|is_regiao_norte|is_regiao_nordeste|is_regiao_sudeste|is_regiao_sul|is_mulher|is_homem|is_raca_branca|is_raca_preta|is_raca_amarela|is_raca_parda|is_raca_indigena|is_escol_analfabeto|is_escol_fundamental|is_escol_medio|is_escol_superior|
+----------+-----+-------+--------+------+------+---------+----------+--------+----------+--------+----------+-----+---------+---------+----------+-------------